In [1]:
from pathlib import Path
import sys
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import torch
import numpy as np
import torch.nn as nn
import pandas as pd

from utils.reproducibility_utils import seed_everything
from utils.data_loading import load_data, load_test_data

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

EXPECTED_ROWS = 49328

Device: cuda


In [ ]:
# EXACT winning architecture
class RNN(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.rnn = nn.RNN(input_size, 64, 1, nonlinearity='relu', batch_first=True)
        self.fc = nn.Linear(64, 64)
        self.out = nn.Linear(64, 1)
        self.act = nn.ReLU()
    def forward(self, x):
        x, _ = self.rnn(x)
        return self.out(self.act(self.fc(x[:, -1, :])))

class WeightedL1(nn.Module):
    def __init__(self, threshold=2.0, slope=1.0):
        super().__init__()
        self.threshold = threshold
        self.slope = slope
    def forward(self, pred, target):
        weights = 1.0 + self.slope * torch.clamp(torch.abs(target) - self.threshold, min=0)
        return (torch.abs(pred - target) * weights).mean()

def predict(model, loader):
    model.eval()
    preds = []
    with torch.no_grad():
        for x, _ in loader:
            preds.append(model(x.to(device)).cpu().numpy())
    return np.concatenate(preds).flatten()

def save_submission(preds, filename):
    assert len(preds) == EXPECTED_ROWS, f"Expected {EXPECTED_ROWS}, got {len(preds)}"
    df = pd.DataFrame({"predicted_sideslip": preds, "id": range(len(preds))})
    df.to_csv(filename, index=False)
    print(f"Saved: {filename} | mean={preds.mean():.4f}, std={preds.std():.4f}")

print("Components ready")

✅ Components ready


In [ ]:
# Load data
print("Loading data...")
train_loader, val_loader, pipeline = load_data(
    train_path="../datasets_challenge/train",
    train_ratio=0.98, batchsize=64,
    columns_to_drop=["sideSlip", "vyCG", "time", "longitude", "latitude"],
    sequence_length=20, threshold=True, threshold_value=3.0,
    threshold_column="vxCG", target="sideSlip", scaling=True, seed=42,
)
test_loader = load_test_data("../datasets_challenge/test", 512, pipeline, load_anonymized_data=True)
input_size = next(iter(train_loader))[0].shape[2]

test_count = sum(x.shape[0] for x, _ in test_loader)
print(f"Data loaded | Input: {input_size} | Test: {test_count}")
assert test_count == EXPECTED_ROWS

Loading data...
../datasets_challenge/train/train_part_001.csv
../datasets_challenge/train/train_part_002.csv
../datasets_challenge/train/train_part_003.csv
../datasets_challenge/train/train_part_004.csv
../datasets_challenge/train/train_part_005.csv
../datasets_challenge/train/train_part_006.csv
../datasets_challenge/train/train_part_007.csv
../datasets_challenge/train/train_part_007.csv
../datasets_challenge/train/train_part_008.csv
../datasets_challenge/train/train_part_009.csv
../datasets_challenge/train/train_part_010.csv
../datasets_challenge/train/train_part_011.csv
../datasets_challenge/train/train_part_008.csv
../datasets_challenge/train/train_part_009.csv
../datasets_challenge/train/train_part_010.csv
../datasets_challenge/train/train_part_011.csv
../datasets_challenge/train/train_part_012.csv
../datasets_challenge/train/train_part_013.csv
../datasets_challenge/train/train_part_012.csv
../datasets_challenge/train/train_part_013.csv
../datasets_challenge/train/train_part_014.c

In [ ]:
# Training functions
def train_two_stage(seed, s1_epochs=30, s2_epochs=40, s2_lr=0.00005, threshold=2.0, slope=1.0):
    seed_everything(seed)
    model = RNN(input_size).to(device)
    
    # Stage 1: L1, lr=0.0003
    opt = torch.optim.Adam(model.parameters(), lr=0.0003)
    crit = nn.L1Loss()
    best, state = np.inf, None
    for _ in range(s1_epochs):
        model.train()
        for x, y in train_loader:
            opt.zero_grad()
            crit(model(x.to(device)), y.to(device).reshape(-1,1)).backward()
            opt.step()
        model.eval()
        val = sum(crit(model(x.to(device)), y.to(device).reshape(-1,1)).item() for x,y in val_loader) / len(val_loader)
        if val < best: best, state = val, model.state_dict().copy()
    model.load_state_dict(state)
    
    # Stage 2: WeightedL1
    opt = torch.optim.Adam(model.parameters(), lr=s2_lr)
    crit2 = WeightedL1(threshold=threshold, slope=slope)
    best2, state2 = np.inf, None
    for _ in range(s2_epochs):
        model.train()
        for x, y in train_loader:
            opt.zero_grad()
            crit2(model(x.to(device)), y.to(device).reshape(-1,1)).backward()
            opt.step()
        model.eval()
        val = sum(nn.L1Loss()(model(x.to(device)), y.to(device).reshape(-1,1)).item() for x,y in val_loader) / len(val_loader)
        if val < best2: best2, state2 = val, model.state_dict().copy()
    model.load_state_dict(state2)
    return model

def train_baseline(seed):
    seed_everything(seed)
    model = RNN(input_size).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=0.0003)
    crit = nn.L1Loss()
    best, state, patience = np.inf, None, 0
    for _ in range(50):
        model.train()
        for x, y in train_loader:
            opt.zero_grad()
            crit(model(x.to(device)), y.to(device).reshape(-1,1)).backward()
            opt.step()
        model.eval()
        val = sum(crit(model(x.to(device)), y.to(device).reshape(-1,1)).item() for x,y in val_loader)
        if val < best: best, state, patience = val, model.state_dict().copy(), 0
        else:
            patience += 1
            if patience > 10: break
    model.load_state_dict(state)
    return model

print("Training functions ready")

✅ Training functions ready


In [ ]:
# Train baseline models
SEEDS = [42, 2024]

print("="*60)
print("TRAINING BASELINE MODELS")
print("="*60)

baseline_preds = [predict(train_baseline(s), test_loader) for s in SEEDS]
baseline_ens = np.mean(baseline_preds, axis=0)
print(f"\nBaseline ready | mean={baseline_ens.mean():.4f}")

📦 TRAINING BASELINE MODELS

✅ Baseline ready | mean=-0.0106

✅ Baseline ready | mean=-0.0106


In [ ]:
# Experiment 1: S1=32, LR=4e-5
print("="*60)
print("COMBO 1: S1=32, LR=4e-5")
print("="*60)

preds = [predict(train_two_stage(s, s1_epochs=32, s2_lr=0.00004, s2_epochs=40, threshold=2.0), test_loader) for s in SEEDS]
ens = np.mean(preds, axis=0)
blend = (baseline_ens + ens) / 2
save_submission(blend, "pred_v16_s132_lr4e5.csv")

🧪 COMBO 1: S1=32, LR=4e-5
✅ pred_v16_s132_lr4e5.csv | mean=-0.0019, std=1.4891
✅ pred_v16_s132_lr4e5.csv | mean=-0.0019, std=1.4891


In [ ]:
# Experiment 2: S1=32, Thresh=2.2
print("="*60)
print("COMBO 2: S1=32, Thresh=2.2")
print("="*60)

preds = [predict(train_two_stage(s, s1_epochs=32, s2_lr=0.00005, s2_epochs=40, threshold=2.2), test_loader) for s in SEEDS]
ens = np.mean(preds, axis=0)
blend = (baseline_ens + ens) / 2
save_submission(blend, "pred_v16_s132_thresh22.csv")

🧪 COMBO 2: S1=32, Thresh=2.2
✅ pred_v16_s132_thresh22.csv | mean=-0.0019, std=1.4948
✅ pred_v16_s132_thresh22.csv | mean=-0.0019, std=1.4948


In [ ]:
# Experiment 3: S1=32, LR=4e-5, Thresh=2.2
print("="*60)
print("COMBO 3: S1=32, LR=4e-5, Thresh=2.2")
print("="*60)

preds = [predict(train_two_stage(s, s1_epochs=32, s2_lr=0.00004, s2_epochs=40, threshold=2.2), test_loader) for s in SEEDS]
ens = np.mean(preds, axis=0)
blend = (baseline_ens + ens) / 2
save_submission(blend, "pred_v16_s132_lr4e5_thresh22.csv")

🧪 COMBO 3: S1=32, LR=4e-5, Thresh=2.2
✅ pred_v16_s132_lr4e5_thresh22.csv | mean=0.0031, std=1.4882
✅ pred_v16_s132_lr4e5_thresh22.csv | mean=0.0031, std=1.4882


In [ ]:
# Experiment 4: S1=32, LR=4e-5, S2 Epochs Sweep
print("="*60)
print("COMBO 4: S1=32, LR=4e-5, S2 Epochs Sweep")
print("="*60)

for ep in [38, 42, 44]:
    print(f"\nS2 Epochs: {ep}")
    preds = [predict(train_two_stage(s, s1_epochs=32, s2_lr=0.00004, s2_epochs=ep, threshold=2.0), test_loader) for s in SEEDS]
    ens = np.mean(preds, axis=0)
    blend = (baseline_ens + ens) / 2
    save_submission(blend, f"pred_v16_s132_lr4e5_{ep}ep.csv")

🧪 COMBO 4: S1=32, LR=4e-5, S2 Epochs Sweep

📦 S2 Epochs: 38
✅ pred_v16_s132_lr4e5_38ep.csv | mean=-0.0010, std=1.4905

📦 S2 Epochs: 42
✅ pred_v16_s132_lr4e5_38ep.csv | mean=-0.0010, std=1.4905

📦 S2 Epochs: 42
✅ pred_v16_s132_lr4e5_42ep.csv | mean=-0.0017, std=1.4948

📦 S2 Epochs: 44
✅ pred_v16_s132_lr4e5_42ep.csv | mean=-0.0017, std=1.4948

📦 S2 Epochs: 44
✅ pred_v16_s132_lr4e5_44ep.csv | mean=0.0008, std=1.4950
✅ pred_v16_s132_lr4e5_44ep.csv | mean=0.0008, std=1.4950


In [ ]:
import glob
print("="*60)
print("GENERATED FILES")
print("="*60)
files = sorted(glob.glob("pred_v16_*.csv"))
for f in files:
    print(f)

📋 GENERATED FILES
pred_v16_s132_lr4e5.csv
pred_v16_s132_lr4e5_38ep.csv
pred_v16_s132_lr4e5_42ep.csv
pred_v16_s132_lr4e5_44ep.csv
pred_v16_s132_lr4e5_thresh22.csv
pred_v16_s132_thresh22.csv
